# FINS 实验流水线

一条 notebook 串起 4 个模块（原 notebook 已 py 化）：

| 步骤       | 模块 | 作用                                                                                           |
|------------|---|------------------------------------------------------------------------------------------------|
| 1 生成配置 | `_load.py` | 按拓扑 / 时序 / 负载随机生成 pipeline cfg → `pipeline/*.json`                             |
| 2 采集     | `_test.py` | 【fins 测试专用】每份 cfg 起 `bin/client` + `bin/server` 跑 `dur_s` 秒 → trace 复制到 `result/`              |
| 3 标准化   | `std.py` | 【fins 测试专用】 用 JSON 的执行用时在 `execute→complete` 内插 `working` 行 → `result_std/` |
| 4 分析画图 | `plot.py` | 核心甘特、核利用率、生命周期分布                                                               |

> 三个数据目录都在**仓库根**：`pipeline/`（配置）、`result/`（原始 trace）、`result_std/`（标准化 trace）。
> 下面所有相对路径都按仓库根解析，不依赖 notebook 的 cwd。

In [3]:
import _test
import _load2json
import _lttng2csv

## 1. 生成配置（`_load.py`）

5 种拓扑（multihop / fork / join / feedback / mixed，均可混入 acc 的 hist 窗口读）
+ 时序（`ptimed`、`period_divisors`）+ 负载（`u`、`H_ms` 等）随机生成，
文件名 `<kind>_u<u>_m<m>_ms<桶>_s<seed>.json`。

In [4]:
CONFIG_RANDOM = {
    "topology": {
        "multihop": {
            "paths": (1, 4),
            "depth": (2, 8),
            "phist": 0.25,
            "hist": (3, 8),
        },

        "fork": {
            "fan": (2, 8),
            "depth": (1, 3),
            "phist": 0.25,
            "hist": (3, 8),
        },

        "join": {
            "fan": (2, 8),
            "depth": (1, 3),
            "phist": 0.25,
            "hist": (3, 8),
        },

        "feedback": {
            "depth": (3, 8),
            "phist": 0.25,
            "hist": (3, 8),
        },

        "mixed": {
            "nseg": (3, 8),
            "pmultihop": 0.25,
            "pfork": 0.25,
            "pjoin": 0.25,
            "pfeedback": 0.25,
        },
    },
    "temporal": {
        "ptimed": 0.35,
        "pthin": 0.0,
        "period_divisors": [1, 2, 4, 5, 10, 20],
    },
    "workload": {
        "workers": [1, 2, 3],
        "mesg_size": [1024],
        "H_ms": 100,
        "utilization": [0.5, 0.6, 0.7, 0.8, 0.9],
        "makespan": [(0, 10), (10, 20), (20, 30), (30, 40), (40, 50), (50, 60), (60, 70), (70, 80),  (80, 90)],
    },

    "search": {
        "n_per": 2,
        "initial_attempts": 1000,
        "refill_attempts": 1000,
        "max_refill_rounds": 5,
        "candidate_factor": 1,
        "seed_base": 20260809,
    },
    "solver": {
        "utilization_tolerance": 1e-8,
        "min_wcet_us": 1000,
        "max_c_ratio": 0.2,
        "c_attempts": 300,
    },
}
CONFIG_SERIAL = {
    # 只生成 multihop。generate_all 默认遍历 5 种拓扑，CONFIG 里只留一个块
    # 会在其余 kind 上 KeyError，必须显式收窄
    "kinds": ["multihop"],
    "topology": {
        "multihop": {
            "paths": (1, 1),      # 单路径
            "depth": (1, 1),      # 固定跳数 → 节点数 = depth + 2（src + sink）
            "phist": 0.0,         # 0 = 纯 relay，不插 usr_acc
        },
    },
    "temporal": {
        "ptimed": 0.0,            # 0 = 非源节点全部 event 触发
        "period_divisors": [1],   # 源节点周期 = H（anchor 强制，此项兜底）
        "pthin": 0.0,             # 不抽稀 → N=1，上下游严格 1:1
    },
    "workload": {
        "mesg_size": [10*1024*1024, 100*1024*1024, 1024*1024*1024],
        "workers": [1, 2, 3],
        # 样本数 = run_time / H_ms（H 也是链的重复周期）
        # 下限受 min_wcet 约束：8 个节点各 ≥1ms → u·m·H ≥ 8ms
        "H_ms": 4000,
        "utilization": [(0,1)], # 0~100%
        "makespan": [(0, 3500)], # 0~100ms
    },
    "search": {
        "n_per": 5,               # 每桶 1 份 → 每个 (u, m) 出 1 份 cfg
        "initial_attempts": 1000,
        "refill_attempts": 1000,
        "max_refill_rounds": 5,
        "candidate_factor": 5,
        "seed_base": 20260809,
    },
    "solver": {
        "utilization_tolerance": 1e-5,
        "min_wcet_us": 1000 * 1000,      # 插件 cfg 单位 µs；须 > 1MB 载荷成本（~290µs）
        "max_c_ratio": 0.9,       # C_i ≤ T_i × 0.2
        "c_attempts": 300,
    },
}

_load2json.generate_all(
    out_dir="tool/pipeline_serial",
    config=CONFIG_SERIAL
)

[MESG] 稳态产出耗时（本机实测估计）: 10485760B→140.40µs  104857600B→1404.00µs  1073741824B→14376.96µs

[PLAN] kinds=1 utilization=1 m=3 mesg=3[10485760, 104857600, 1073741824] → 求解 3 次，落盘文件数 = 求解次数 × 每档 n_per 文件数 × 3 档

[CONFIG] multihop utilization=(0, 1) m=1
[PHASE-1] kind=multihop utilization=(0, 1) m=1 attempts=1000
[PHASE-1] generated=25
[SUCCESS] kind=multihop utilization=(0, 1) m=1 total=5
[SAVED] 5 files mesg=10485760B
[SAVED] 5 files mesg=104857600B
[SAVED] 5 files mesg=1073741824B

[CONFIG] multihop utilization=(0, 1) m=2
[PHASE-1] kind=multihop utilization=(0, 1) m=2 attempts=1000
[PHASE-1] generated=25
[SUCCESS] kind=multihop utilization=(0, 1) m=2 total=5
[SAVED] 5 files mesg=10485760B
[SAVED] 5 files mesg=104857600B
[SAVED] 5 files mesg=1073741824B

[CONFIG] multihop utilization=(0, 1) m=3
[PHASE-1] kind=multihop utilization=(0, 1) m=3 attempts=1000
⚠️ utilization=(0, 1) 与可行域 [0.2500, 0.3333] 相交后被夹为 [0.2500, 0.3333]（节点 3 个, m=3）
[RESOLVE] utilization=(0, 1) → 取样于 [0.2500, 0.3333]
[PHAS

['/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u82_m1_ms00_msg10485760_s20686443.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u76_m1_ms00_msg10485760_s20686429.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u84_m1_ms00_msg10485760_s20686409.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u82_m1_ms00_msg10485760_s20686450.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u87_m1_ms00_msg10485760_s20686421.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u82_m1_ms00_msg104857600_s20686443.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u76_m1_ms00_msg104857600_s20686429.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u84_m1_ms00_msg104857600_s20686409.json',
 '/home/jenny/Documents/GitHub/fins/tool/pipeline_serial/multihop_u82_m1_ms00_msg104857600_s20686450.json',
 '/home/jenny/Documents/GitHub/fi

## 2. 采集（`_test.py`）

对每份 cfg：起 `client` + 发配置 + 跑 `dur_s` 秒 → 终止 → 把 `tool/temp/tracing.csv`
复制成 `result/<cfg名>.csv`（原件保留，只搬原始 trace，不算指标）。

正式实验要走独占核：`cores="1-6"` 会改用 `sudo tool/client.sh <cores> <workers>`（需要 root）。

In [4]:
# 指定你的目录
my_cfg_dir = "tool/pipeline_serial"
my_result_dir = "tool/FINS_serial"

# 一键运行（cpu_offset=1 代表核心从 1 开始排，如果 m=3 就会自动分配核 "1-3"）
_test.run_all(
    target_cfg_dir=my_cfg_dir,
    target_result_dir=my_result_dir,
    run_time=400.0,     # 运行时间
    cpu_offset=1      # 起始核号（避开核心 0 给系统）
)

🔒 该评测脚本需要 root 权限来配置 Cgroup 与绑定核心
🧹 跑前守卫：检查残留 client / 端口占用...
🚀 开始智能批量评测（自动从文件名匹配 m）
   📂 输入配置目录: tool/pipeline_serial
   📁 结果输出目录: tool/FINS_serial
   📊 发现测试用例: 93 个
   ⏭️ 断点续跑: 开（已有的跳过；强制全跑把 SKIP_DONE 置 False）

进度 [1/93]: multihop_u26_m3_ms00_msg104857600_s20752497.json
  ⏭️ 跳过（已有完整结果: tool/FINS_serial/multihop_u26_m3_ms00_msg104857600_s20752497_221231）

进度 [2/93]: multihop_u26_m3_ms00_msg104857600_s20752505.json
  ⏭️ 跳过（已有完整结果: tool/FINS_serial/multihop_u26_m3_ms00_msg104857600_s20752505_221916）

进度 [3/93]: multihop_u26_m3_ms00_msg104857600_s20752512.json
  ⏭️ 跳过（已有完整结果: tool/FINS_serial/multihop_u26_m3_ms00_msg104857600_s20752512_222601）

进度 [4/93]: multihop_u26_m3_ms00_msg10485760_s20752497.json
  ⏭️ 跳过（已有完整结果: tool/FINS_serial/multihop_u26_m3_ms00_msg10485760_s20752497_223246）

进度 [5/93]: multihop_u26_m3_ms00_msg10485760_s20752505.json
  ⏭️ 跳过（已有完整结果: tool/FINS_serial/multihop_u26_m3_ms00_msg10485760_s20752505_223931）

进度 [6/93]: multihop_u26_m3_ms00_msg10485760_s20752512.json
 

# 3. 转义（tran）
1. lttng 2 timeline
2. lttng 2 overhead

In [5]:
_lttng2csv.run_export(
    results_dir="./FINS_serial/a",
    outdir="./FINS_serial",
    after_us=2000 * 1000,
    preempt_all=True,
    cpu_offset=1,
)


开始处理 50 个实验 -> ./FINS_serial/  输出: timeline, preempt
  [跳过] multihop_u26_m3_ms00_msg104857600_s20752497_001603: 没有 trace/ 子目录  cpus=1-3(m=3)
  [跳过] multihop_u26_m3_ms00_msg104857600_s20752497_215917: 没有 trace/ 子目录  cpus=1-3(m=3)
  [成功] multihop_u26_m3_ms00_msg104857600_s20752497_221231  t0=55168323725  cpus=1-3(m=3)  ->  multihop_u26_m3_ms00_msg104857600_s20752497_221231_timeline.csv (178910 行) | multihop_u26_m3_ms00_msg104857600_s20752497_221231_preempt.csv (0 行)
  [成功] multihop_u26_m3_ms00_msg104857600_s20752505_221916  t0=460264060005  cpus=1-3(m=3)  ->  multihop_u26_m3_ms00_msg104857600_s20752505_221916_timeline.csv (178052 行) | multihop_u26_m3_ms00_msg104857600_s20752505_221916_preempt.csv (0 行)
  [成功] multihop_u26_m3_ms00_msg104857600_s20752512_222601  t0=865330595098  cpus=1-3(m=3)  ->  multihop_u26_m3_ms00_msg104857600_s20752512_222601_timeline.csv (178724 行) | multihop_u26_m3_ms00_msg104857600_s20752512_222601_preempt.csv (0 行)
  [成功] multihop_u26_m3_ms00_msg10485760_s20752497

KeyboardInterrupt: 